In [1]:
import os
from dotenv import load_dotenv

import textwrap



def pretty_print(*args, **kwargs):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80), **kwargs)
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

result=load_dotenv('./openApiKey.env')  # reads .env file in the current directory

#print("Loaded:",result)
api_key = os.getenv("OPENAI_API_KEY")
print(os.getcwd())

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

/Users/anupagarwal/Documents/PersonalFolder/ScalerAICourse/Practicals/workingwithllms
API key loaded successfully.


In [3]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")
models=client.models.list()
for m in models.data:
    pretty_print(m.id)
    break

OpenAI client ready.
text-embedding-ada-002


In [4]:
from openai import OpenAI

client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

MODEL  = "gpt-5-nano"  

OpenAI client ready.


In [5]:
# The model has NO access to live data — what does it do?

response = client.responses.create(
    model=MODEL,
    input="What is the weather in bengaluru right now? Give me the exact temperature.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
print(response.output_text)

I don’t have real-time weather data access. You can check a weather app or website (e.g., Weather.com, IMD, or your smartphone weather widget) for the exact current temperature in Bengaluru. If you’d like, I can guide you on how to find it quickly.


In [7]:
# The model PREDICTS math — it doesn't COMPUTE it

response = client.responses.create(
    model=MODEL,
    input="What is 1247 * 83 + 19 / 3.7? Give me answer in one line (max 10 words), the exact number, upto 2 decimal places.",
    reasoning={"effort": "minimal"},   
    text={"verbosity": "low"}
)
print("Model says:", response.output_text)

# What Python actually computes
import math
actual = 1247 * 83 + 19 / 3.7
print(f"Actual answer: {actual}")

Model says: 103,?
Actual answer: 103506.13513513513


In [8]:
# Define a simple "add" tool — we'll explain the structure in detail next
add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

# how to tell the model what function exists, and what argument it takes. 

# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="What is 7 + 12?",
    tools=[add_tool],
)



# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: add
  Arguments: {"a":7,"b":12}
  Call ID: call_E59TDb43NXyvhHvmbhSKy5Og
 Response:


In [9]:
def add(a, b):
    print(f"Adding {a} and {b} together...")
    return a + b

    
add(**{"a": 7, "b": 12})

Adding 7 and 12 together...


19

In [10]:
# Ask the model to add — but give it the tool
response = client.responses.create(
    model=MODEL,
    instructions="Use the add tool for any math. Never compute math yourself.",
    input="Why is Earth round?",
    tools=[add_tool],
)


# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: message
 Response: Earth isn’t a perfect ball, but it is round in a big-picture sense.
Here’s why:  - Gravity pulls matter toward a center. If a body can rearrange
itself, gravity tends to make it as compact as possible, which is a sphere. -
But Earth also spins. The rotation creates a centrifugal force that pushes
outward most at the equator, causing the planet to bulge slightly there. - The
result is an oblate spheroid: a shape a bit wider at the equator than from pole
to pole.  Some quick numbers (approx): - Equatorial radius: about 6,378 km -
Polar radius: about 6,357 km - Difference about 21 km - Flattening (how Oblate
it is) is about 1/298  So while mountains and trenches make local unevenness,
the global shape is round because gravity tries to compress matter into a
sphere, with a bit of flattening due to rotation. If you’d like, I can go into
more detail about the geoid or how satell

In [13]:
# Adding more tools doesn't change the model's behavior if the question doesn't require them
import json

add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a,b):
    return a + b 
def subtract(a,b):
    return a-b
def multiply(a,b):
    return a*b
def divide(a,b):
    if b==0:
        return "Error : Division by zero not possible."
    return a/b

response=client.responses.create(
    model=MODEL,
    instructions="Use the tools for any and every kind of math. Never compute math yourself.",
    input="what is 1247*83 + 19 /3.7 ?",
    tools=[add_tool,sub_tool,mul_tool,div_tool],
    reasoning={"effort":"high"}
)


print("Output items from model:")
print("-"*40)
for item in response.output:
    print(f"Type: {item.type}")
    if item.type=="function_call":
        print(f" Function name: {item.name}")
        print(f" Arguents: {item.arguments}")
        print(f" Call ID : {item.call_id}")
pretty_print(f" Response : {response.output_text}")


Output items from model:
----------------------------------------
Type: reasoning
Type: function_call
 Function name: multiply
 Arguents: {"a":1247,"b":83}
 Call ID : call_GFkdQNhzMbS8ZJ6kBzA5hJc2
Type: function_call
 Function name: divide
 Arguents: {"a":19,"b":3.7}
 Call ID : call_bCebCeZDbvLHiAmdY4RYPvrs
 Response :


In [14]:
import json
DISPATCH = { 
    "add":add,
    "subtract":subtract,
    "multiply":multiply,
    "divide":divide
}

for i,item in enumerate(response.output):
    if item.type=="function_call":
        func_name=item.name
        arguments=item.arguments
        call_id=item.call_id
        print(f" func_name: {func_name}, arguments :{arguments} , call_id: {call_id}")
        if func_name in DISPATCH:
            func=DISPATCH[func_name]
            mod_args=json.loads(arguments)
            result=func(**(mod_args))
            print(f" Result of {func_name} with arguments {arguments} : {result}")
        else:
            print(f" Unknown function call method :{func_name}")
pretty_print(f" Model outputs :{response.output_text}")
            

 func_name: multiply, arguments :{"a":1247,"b":83} , call_id: call_GFkdQNhzMbS8ZJ6kBzA5hJc2
 Result of multiply with arguments {"a":1247,"b":83} : 103501
 func_name: divide, arguments :{"a":19,"b":3.7} , call_id: call_bCebCeZDbvLHiAmdY4RYPvrs
 Result of divide with arguments {"a":19,"b":3.7} : 5.135135135135135
 Model outputs :


In [15]:
# Define a get_weather tool
weather_tool = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current temperature for a given city.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "City name, e.g. 'Tel Aviv', 'London'"
            },
        },
        "required": ["location"],
        "additionalProperties": False,
    },
    "strict": True,
}


# Same question as before — but now the model has a tool!
response = client.responses.create(
    model=MODEL,
    input="What's the weather in Paris right now?",
    tools=[weather_tool],
)

# Let's inspect what came back
print("Output items from the model:")
print("-" * 40)
for item in response.output:
    print(f"  Type: {item.type}")
    if item.type == "function_call":
        print(f"  Function name: {item.name}")
        print(f"  Arguments: {item.arguments}")
        print(f"  Call ID: {item.call_id}")
pretty_print(" Response: " + response.output_text)

Output items from the model:
----------------------------------------
  Type: reasoning
  Type: function_call
  Function name: get_weather
  Arguments: {"location":"Paris"}
  Call ID: call_F4k9GvUw8NpWgbrUeH426arZ
 Response:


In [17]:
# Our fake weather function (in production, this would call a real API)
def get_weather(location):
    fake_data = {"Tel Aviv": "28°C, sunny", "Paris": "18°C, cloudy", "London": "14°C, rain"}
    return fake_data.get(location, f"No data for {location}")

# Step 1: Ask with the weather tool
response = client.responses.create(
    model=MODEL,
    input="What's the weather like in Tel Aviv and London?",
    tools=[weather_tool],
)

# Step 2 & 3: Find all function calls, execute each
print("Model requested these calls:")
new_input = list(response.output)
#print(new_input)

for item in response.output:
    if item.type == "function_call":
        args = json.loads(item.arguments)
        result = get_weather(**args)
        print(f"  → {item.name}({args}) = {result}")
        
        # Step 4: Append our result
        new_input.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": result,
        })
       # print(new_input)

# Step 5: Model composes final answer with real data
final = client.responses.create(
    model=MODEL,
    input=new_input,
    tools=[weather_tool],
)
print(f"\nFinal answer:\n{final.output_text}")

Model requested these calls:
  → get_weather({'location': 'Tel Aviv'}) = 28°C, sunny
  → get_weather({'location': 'London'}) = 14°C, rain

Final answer:
Here’s the current snapshot:

- Tel Aviv: 14°C, rain
- London: 28°C, sunny

Want an hourly forecast or a 5-day outlook for either city?


In [22]:
# Adding more tools doesn't change the model's behavior if the question doesn't require them

add_tool = {
    "type": "function",
    "name": "add",
    "description": "Add two numbers together and return the sum.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

sub_tool = {
    "type": "function",
    "name": "subtract",
    "description": "Subtract two numbers and return the difference.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

mul_tool = {
    "type": "function",
    "name": "multiply",
    "description": "Multiply two numbers together and return the product.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "First number"},
            "b": {"type": "number", "description": "Second number"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}


div_tool = {
    "type": "function",
    "name": "divide",
    "description": "Divide two numbers and return the answer.",
    "parameters": {
        "type": "object",
        "properties": {
            "a": {"type": "number", "description": "Numerator (dividend)"},
            "b": {"type": "number", "description": "Denominator (divisor)"},
        },
        "required": ["a", "b"],
        "additionalProperties": False,
    },
    "strict": True,
}

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    if b == 0:
        return "Error: Division by zero"
    return a / b


DISPATCH = {
    "add": add,
    "subtract": subtract,
    "multiply": multiply,
    "divide": divide,
}

TOOLS = [add_tool, sub_tool, mul_tool, div_tool]
DEV_POLICY = "Use the tools for any math. Never compute math yourself."

def getanswers_fromModel_and_processing(user_question:str)->str:
     print("══════════════════════════════════════════════════════")
     print("START: User question")
     print("  ", user_question)
     print("══════════════════════════════════════════════════════")
     
     # Round 1: seed conversation with developer policy + user question
     resp = client.responses.create(
        model=MODEL,
        input=[
            {"role": "developer", "content": DEV_POLICY},
            {"role": "user", "content": user_question},
        ],
        tools=TOOLS
     )
     round_num=1
     while True:
         print(f"\n══════════════════════════════════════════════════════")
         print(f"ROUND {round_num}: Model response items")
         print("══════════════════════════════════════════════════════")
         
         for i,item in enumerate(resp.output):
             print(f"[{i}] type={item.type}")
             if item.type =="function_call":
                 print(f"    name     = {item.name}")
                 print(f"    call_id  = {item.call_id}")
                 print(f"    arguments(raw JSON string) = {item.arguments}")
             elif item.type=="message":
                 # Some SDKs represent assistant text as message items
                # output_text is the easiest way to get the final combined text.
                 try:
                    print(f"    content = {item.content}")
                 except Exception:
                    pass
         calls=[item for i,item in enumerate(resp.output) if item.type=="function_call"]
         if not calls:
             print("\n✅ No function calls. Final assistant text:")
             print(resp.output_text)
             return resp.output_text
         print("\n→ Model requested tool calls:")
         for call in calls:
                 print(f"  - {call.name}({call.arguments})  [call_id={call.call_id}]")
                 
         tool_outputs=[]
         print("\n→ Executing tools locally (your server/app):")
         
         for call in calls:
             fn=DISPATCH.get(call.name)
             args=json.loads(call.arguments)
             try:
                 result=fn(**args)
                 payload = {"ok": True, "result": result}
                 print(f"  ✓ {call.name}(**{args}) -> {result}")
             except Exception as e:
                 payload = {"ok": False, "error": str(e)}
                 print(f"  ✗ {call.name}(**{args}) -> ERROR: {e}")
                     
             tool_outputs.append({
                     "type":"function_call_output",
                     "call_id":call.call_id,
                     "output":json.dumps(payload)
             })
                 
             print("\n→ Sending tool outputs back to the model:")
             for out in tool_outputs:
                    pretty_print(out)
                    
         resp=client.responses.create(
                     model=MODEL,
                     previous_response_id=resp.id,
                     input=tool_outputs,
                     tools=TOOLS,
                     reasoning={"effort":"high"}
         )
         round_num+=1
    


# Example
final_text = getanswers_fromModel_and_processing("What is the value of 17 -24 / 6 *4 +8 ?")
#final_text = getanswers_fromModel_and_processing("I have 15 oranges, 10 apples, 3 bananas and 4 avocados.  How many fruits do I have now?")
print(final_text)
                 
      

══════════════════════════════════════════════════════
START: User question
   What is the value of 17 -24 / 6 *4 +8 ?
══════════════════════════════════════════════════════

══════════════════════════════════════════════════════
ROUND 1: Model response items
══════════════════════════════════════════════════════
[0] type=reasoning
[1] type=function_call
    name     = divide
    call_id  = call_mZHgeqmTx3mlrl8moc6gCpCR
    arguments(raw JSON string) = {"a":24,"b":6}

→ Model requested tool calls:
  - divide({"a":24,"b":6})  [call_id=call_mZHgeqmTx3mlrl8moc6gCpCR]

→ Executing tools locally (your server/app):
  ✓ divide(**{'a': 24, 'b': 6}) -> 4.0

→ Sending tool outputs back to the model:
{'type': 'function_call_output', 'call_id': 'call_mZHgeqmTx3mlrl8moc6gCpCR',
'output': '{"ok": true, "result": 4.0}'}

══════════════════════════════════════════════════════
ROUND 2: Model response items
══════════════════════════════════════════════════════
[0] type=reasoning
[1] type=function_call
